# Limitierung der Regelparameter

Jonas Frei, 25.09.2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Die Limitierung der Regelparameter wird am Beispiel einer geschwindigkeitsgeregelten Punktmasse mit Masse $m$ betrachtet.

## Systemdefinition

### Masse

In [ ]:
m = 1

### Regelfrequenz und Regeltakt

In [ ]:
fTask = 1000
Ts = 1/fTask
Ts

### Dämpfung des Reglers

In [ ]:
D = 0.7

### Bandbreite

Drei Varianten der Bandbreite $\omega_0$ werden verglichen:

1. Steifigkeitsgrenze: $\omega_0 = f_{Task}/(2D)$
2. Ausgleich über mehrere Regeltakte: $\omega_0 = f_{Task}/(4{,}6D)$
3. Überschiessen: $\omega_0 = f_{Task}/D$

In [ ]:
varianten = {
    1: ("Steifigkeitsgrenze", fTask/2/D),
    2: ("Ausgleich über mehrere Regeltakte", fTask/4.6/D),
    3: ("Überschiessen", fTask/D),
}

### Proportional- und Integralverstärkung

$k_p = 2D\omega_0$, $k_i = \omega_0^2$

## Simulation

![image](images/blockschaltbild.svg)

- **Regler** (zeitdiskret, Regeltakt `Ts`): kinematischer Regler, liefert aus dem Fehler $e$ die Kraft $F$ (PI-Struktur, $F=m\cdot(k_p e + k_i\!\int\! e\,dt)$).
- **Strecke** (zeitkontinuierlich): Punktmasse $1/(m\cdot s)$, integriert die vom Regler gehaltene Kraft $F$ zur Geschwindigkeit $v(t)$.

In [ ]:
def step(t, step_time=0.01, amplitude=1.0, offset=0.0):
    return amplitude+offset if t >= step_time else offset


def regler(e, E, kp, ki):
    """Zeitdiskreter kinematischer Regler (Regeltakt Ts).

    Liefert die Kraft F, die an die Strecke übergeben wird, sowie den
    aktualisierten Integratorzustand E.
    """
    a_cmd = kp*e + ki*E   # PI: intern berechnete Beschleunigung
    F = m*a_cmd           # Reglerausgang nach Newton: F = m*a
    E_neu = E + Ts*e      # diskrete Integration des Fehlers
    return F, E_neu


def strecke(v, F, m, dt):
    """Zeitkontinuierliche Punktmasse: dv/dt = F/m, ein Euler-Schritt.

    F wird während dt als konstant angenommen (Halten durch den Regler
    bis zum nächsten Regeltakt).
    """
    return v + dt*F/m


def simulate(om0, T_end=0.04, n_substeps=10):
    kp = 2*D*om0
    ki = om0**2
    dt = Ts/n_substeps
    n_ticks = int(round(T_end/Ts))

    v = 0.0
    E = 0.0

    t_arr = [0.0]
    v_arr = [0.0]

    for tick in range(n_ticks):
        # Regler: läuft einmal pro Regeltakt Ts
        e = step(t = tick * Ts) - v
        F, E = regler(e, E, kp, ki)

        # Strecke: läuft kontinuierlich
        for substep in range(1, n_substeps + 1):
            t = tick * Ts + substep * dt
            v = strecke(v, F, m, dt)
            
            t_arr.append(t)
            v_arr.append(v)

    return np.array(t_arr), np.array(v_arr)

## Plotten

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for steifigkeit, (label, om0) in varianten.items():
    t, v = simulate(om0)
    ax.plot(t, v, label=f"{steifigkeit}: {label}")

v_soll = []
for t_ in t:
    v_soll.append(step(t_))
ax.plot(t, v_soll, color="black", linestyle="--", linewidth=1, label="Sollgeschwindigkeit")
ax.set_xlabel("Zeit [s]")
ax.set_ylabel("Geschwindigkeit [m/s]")
ax.set_title("Geschwindigkeitsregelung -- Einfluss der Bandbreite")
ax.legend()
ax.grid(True)
plt.show()

Die Steifigkeitsgrenze markiert theoretisch den Punkt, an dem der Regelfehler innerhalb eines einzigen Regeltakts ausgeglichen würde, dennoch beobachten wir in der Simulation ein Überschwingen. Dies theoretisch Erwartete Verhalten gilt tatsächlich nur bei einer trägheitslosen Strecke. Das in der Simulation sichtbare Überschwingen entsteht dadurch, dass die Punktmasse aufgrund ihrer Trägheit sich auch dann noch weiter Bewegt, wenn der Regler die Sollkraft bereits zurücknimmt.